# nb_10 — FactStockMovement & FactInventorySnapshot

**Purpose:** Build inventory fact tables from `fact_stock_movement.csv`.

## Overview
1. **FactStockMovement** — Raw movement records joined to DimProduct, DimWarehouse, DimDate surrogate keys
2. **FactInventorySnapshot** — Daily running on-hand balance per product+warehouse computed via window function
   - Partitioned by `product_key` for efficient per-product queries

**Prerequisites:** nb_04 (DimProduct), nb_08 (DimWarehouse), nb_01 (DimDate) must be run first.

In [ ]:
from pyspark.sql.functions import col, to_date, sum as _sum
from pyspark.sql import Window

# Load dimension surrogate key lookups
dim_product   = spark.sql("SELECT product_id, product_key FROM DimProduct WHERE is_current = true")
dim_warehouse = spark.sql("SELECT warehouse_id, warehouse_id AS warehouse_key FROM DimWarehouse")
date_map      = spark.sql("""
    SELECT date_key,
           CAST(CONCAT(SUBSTR(CAST(date_key AS STRING),1,4),'-',
                       SUBSTR(CAST(date_key AS STRING),5,2),'-',
                       SUBSTR(CAST(date_key AS STRING),7,2)) AS DATE) AS cal_date
    FROM DimDate
""")

print("Dimensions loaded.")

In [ ]:
# Read raw stock movement CSV
df_move = spark.read.csv(
    "Files/seed_data/fact_stock_movement.csv",
    header=True,
    inferSchema=True
)

print(f"Stock movement rows: {df_move.count()}")
df_move.printSchema()

In [ ]:
# Resolve surrogate keys
df_move = df_move.withColumn("movement_date_parsed", to_date(col("movement_date")))

df_move_keyed = (
    df_move
    .join(dim_product,   on="product_id",   how="left")
    .join(dim_warehouse, on="warehouse_id", how="left")
    .join(
        date_map
        .withColumnRenamed("cal_date", "movement_date_parsed")
        .withColumnRenamed("date_key", "date_key"),
        on="movement_date_parsed",
        how="left"
    )
)

print("Surrogate keys resolved.")

In [ ]:
# Write FactStockMovement
(
    df_move_keyed.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("FactStockMovement")
)

print("FactStockMovement written.")
spark.sql("SELECT COUNT(*) AS row_count FROM FactStockMovement").show()

In [ ]:
# FactInventorySnapshot — running on-hand balance per product+warehouse
w = (
    Window
    .partitionBy("product_key", "warehouse_key")
    .orderBy("date_key")
    .rowsBetween(Window.unboundedPreceding, 0)
)

df_snap = df_move_keyed.withColumn("on_hand_qty", _sum("quantity_delta").over(w))

(
    df_snap.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("product_key")
    .saveAsTable("FactInventorySnapshot")
)

print("FactInventorySnapshot written.")
spark.sql("SELECT COUNT(*) AS row_count FROM FactInventorySnapshot").show()
spark.sql("SELECT product_key, warehouse_key, date_key, on_hand_qty FROM FactInventorySnapshot ORDER BY product_key, date_key LIMIT 10").show()